# HAEM5 Reporter — regenerate the internal test-set predictions (no training)

Rebuilds the **same 70/15/15 split** as the training notebook (same datasets, same seed 42, same de-duplication and caps), runs the **deployed white-cell model** (downloaded automatically from GitHub) on the validation and test images, and saves every prediction. It then computes the statistics the report could not compute before: bootstrap 95% CIs for accuracy, κ and macro-F1, blast AUC, and blast sensitivity/specificity/PPV at the deployed 0.15 threshold — with and without 4-rotation averaging (as the app does).

**Steps:** *Runtime › Change runtime type › T4 GPU* → *Runtime › Run all*. Paste a Kaggle API token when asked (revoke it afterwards). About 45–60 min (mostly downloading the three datasets). At the end, a file **haem5_test_predictions_results.zip** downloads — send it back to Claude.

In [ ]:
# 1. Install only what is missing (Kaggle and Colab already include most packages)
import importlib.util, subprocess, sys, socket
need = [p for p, m in [("timm", "timm"), ("onnx", "onnx"), ("onnxruntime", "onnxruntime"), ("kagglehub", "kagglehub"),
                       ("scikit-learn", "sklearn"), ("seaborn", "seaborn")] if importlib.util.find_spec(m) is None]
def online():
    try: socket.create_connection(("pypi.org", 443), timeout=5).close(); return True
    except OSError: return False
if not online():
    raise SystemExit("❌ No internet. On Kaggle: right panel › Session options (or Settings) › Internet → ON "
                     "(needs a phone-verified account), then run this cell again.")
if need:
    print("Installing:", need)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])
print("✓ Packages ready")

In [ ]:
# 2. Configuration — edit zip names to match what you uploaded to Drive
# EASIEST (Colab): download the dataset straight from Kaggle inside Colab — no manual upload needed.
KAGGLE_DATASETS = ["walkersneps/aml-cytomorphology-lmu",            # blasts + AML-patient cells (Munich)
                   "orvile/microscopic-peripheral-blood-cell-images",  # Barcelona PBC: platelets, erythroblasts, IG stages
                   "andrewmvd/bone-marrow-cell-classification"]         # Munich bone marrow: plasma, hairy, immature lymphoid, proerythroblasts
USE_KAGGLE_DOWNLOAD = True

DATASETS = [
    # Optional extra zips already in your Google Drive (path, short name). Missing ones are skipped.
    # ("/content/drive/MyDrive/PBC_dataset_normal_DIB.zip", "PBC"),
    # ("/content/drive/MyDrive/Cappstone_project_B.zip", "CAPSTONE"),     # optional (no blasts)
]

# App vocabulary (order = model output order). Do not rename — the app relies on these keys.
CLASSES = ["blast", "promyelocyte", "myelocyte", "metamyelocyte", "band_neutrophil",
           "segmented_neutrophil", "eosinophil", "basophil", "monocyte", "lymphocyte",
           "atypical_lymphocyte", "erythroblast", "platelet", "smudge",
           "plasma_cell", "hairy_cell", "immature_lymphocyte", "proerythroblast", "artefact"]

# Folder-name (or filename-prefix) codes -> app class.  Matching is case-insensitive.
# AML-Cytomorphology_LMU codes (Matek 2019). VERIFY with the sample grid in cell 6:
LABEL_MAP = {
    "MYO": "blast",          # myeloblast
    "MOB": "blast",          # monoblast
    "PMO": "promyelocyte",
    "PMB": "promyelocyte",   # bilobed promyelocyte
    "MYB": "myelocyte",
    "MMZ": "metamyelocyte",
    "NGB": "band_neutrophil",
    "NGS": "segmented_neutrophil",
    "EOS": "eosinophil",
    "BAS": "basophil",
    "MON": "monocyte",
    "LYT": "lymphocyte",
    "LYA": "atypical_lymphocyte",
    "EBO": "erythroblast",
    "KSC": "smudge",
    # Munich bone-marrow codes (Matek 2021):
    "BLA": "blast", "PLM": "plasma_cell", "HAC": "hairy_cell", "LYI": "immature_lymphocyte",
    "PEB": "proerythroblast", "ART": "artefact",
    # Barcelona PBC / Capstone filename prefixes and folder names:
    "BA": "basophil", "EO": "eosinophil", "LY": "lymphocyte", "MO": "monocyte",
    "SNE": "segmented_neutrophil", "BNE": "band_neutrophil", "NEUTROPHIL": "segmented_neutrophil",
    "MY": "myelocyte", "MMY": "metamyelocyte", "PMY": "promyelocyte",
    "ERB": "erythroblast", "PLATELET": "platelet", "PLT": "platelet",
}
SKIP_CODES = {"IG", "NIF", "OTH", "ABE", "FGC"}   # pooled IG; not identifiable; other; too few abnormal eos / faggot cells

QUICK = False                # True = tonight's fast run (~45 min). Set False later for the full run (adds bone-marrow set, 15 epochs).
TEST_FRAC = 0.15             # held out until the very end (cell 9)
VAL_FRAC  = 0.15             # used each epoch to pick the best checkpoint
MAX_PER_CLASS = 3000          # cap very large classes (e.g. segmented neutrophils) to speed training
MODEL_NAME  = "efficientnet_b0"
IMG_SIZE    = 224
BATCH_SIZE  = 64
EPOCHS      = 15
LR          = 3e-4
SEED        = 42
OUT_DIR     = "/content/haem5_model"
if QUICK:
    KAGGLE_DATASETS = [k for k in KAGGLE_DATASETS if "bone-marrow" not in k]
    MAX_PER_CLASS, EPOCHS = 1500, 8
    print("⚡ QUICK mode: blood datasets only, ≤1500 images/class, 8 epochs")

# --- Auto-detect Kaggle: datasets attached with "Add Input" are used directly (no zip, no Drive) ---
import os, glob as _g
import sys as _sys
IN_COLAB = "google.colab" in _sys.modules or "COLAB_RELEASE_TAG" in os.environ
ON_KAGGLE = (not IN_COLAB) and (bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle/working"))
if ON_KAGGLE:
    DATASETS = [(d, os.path.basename(d.rstrip("/"))) for d in sorted(_g.glob("/kaggle/input/*/")) + sorted(_g.glob("/kaggle/input/*/*/"))
                if any(os.scandir(d))]
    # keep only top-level dataset folders (avoid double counting nested paths)
    tops = []
    for d, n in DATASETS:
        if not any(d.startswith(t) for t, _ in tops): tops.append((d, n))
    DATASETS = tops
    OUT_DIR = "/kaggle/working/haem5_model"
print("Running on", "Kaggle" if ON_KAGGLE else "Colab", "| datasets:", [n for _, n in DATASETS])

In [ ]:
# 3. Imports
import os, re, glob, zipfile, hashlib, json, time, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
import torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, cohen_kappa_score,
                             classification_report, confusion_matrix)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| classes:", len(CLASSES))
if device.type == "cpu":
    # No GPU available: switch to a lighter model and a smaller sample so training still finishes (about 1–2 h).
    MODEL_NAME, IMG_SIZE, BATCH_SIZE, EPOCHS, MAX_PER_CLASS = "mobilenetv3_large_100", 160, 32, 6, 700
    torch.set_num_threads(os.cpu_count() or 2)
    print(f"⚠ No GPU — CPU mode: {MODEL_NAME}, {IMG_SIZE}px, {EPOCHS} epochs, ≤{MAX_PER_CLASS} images/class. Results will be somewhat lower than with a GPU.")

In [ ]:
# 4. Transforms — the app reproduces eval_tf exactly: resize to 224x224, RGB, ImageNet mean/std
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(180), transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
    transforms.ColorJitter(0.15, 0.15, 0.10, 0.02),   # stain / scanner variation
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

In [ ]:
# 5. Load datasets, map labels, remove duplicates, split
def kaggle_download(slug):
    import kagglehub
    try:
        return kagglehub.dataset_download(slug)
    except Exception as first:
        if ON_KAGGLE:
            print(f"  ⚠ Could not fetch {slug} ({type(first).__name__}). Attach it with + Add Input, or ignore it."); return None
        # Fall back to your Kaggle API key: Colab Secrets (key icon) KAGGLE_USERNAME / KAGGLE_KEY, or upload kaggle.json
        try:
            from google.colab import userdata
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME"); os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        except Exception:
            tok = input("Kaggle needs your API token. Paste it here and press Enter (or just press Enter to upload kaggle.json): ").strip()
            if tok:
                os.environ["KAGGLE_API_TOKEN"] = tok          # new-style token (Kaggle › Settings › API › Create New Token)
                if ":" in tok: os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = tok.split(":", 1)
                uname = input("Your Kaggle username (shown on your Kaggle profile): ").strip()
                if uname: os.environ["KAGGLE_USERNAME"] = uname; os.environ.setdefault("KAGGLE_KEY", tok)
            else:
                from google.colab import files
                up = files.upload(); kj = json.loads(next(iter(up.values())))
                os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = kj["username"], kj["key"]
        return kagglehub.dataset_download(slug)

if USE_KAGGLE_DOWNLOAD:
    have = {n.lower() for _, n in DATASETS}
    for slug in KAGGLE_DATASETS:
        name = slug.split("/")[-1]
        if name.lower() in have:
            print("Already attached:", name); continue
        print("Downloading", slug, "from Kaggle (a few minutes)…")
        path = kaggle_download(slug)
        if path:
            print("  →", path); DATASETS = DATASETS + [(path, name)]
if not ON_KAGGLE and any(p.startswith("/content/drive") for p, _ in DATASETS):
    from google.colab import drive
    drive.mount("/content/drive")

IMG_EXT = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")
def code_to_class(folder, fname):
    keys = []
    m = re.match(r"([A-Za-z]+)", os.path.basename(fname))
    if m: keys.append(m.group(1).upper())          # filename prefix first (e.g. PMY_/MY_/MMY_ inside PBC's "ig" folder)
    keys.append(os.path.basename(folder).upper())
    for k in keys:
        if k in SKIP_CODES: return None
        if k in LABEL_MAP: return LABEL_MAP[k]
    # fuzzy folder words (e.g. 'training data eosinophil')
    f = os.path.basename(folder).lower()
    for word, cls in [("metamyelo", "metamyelocyte"), ("promyelo", "promyelocyte"), ("myelocyte", "myelocyte"),
                      ("baso", "basophil"), ("eosin", "eosinophil"), ("lympho", "lymphocyte"), ("mono", "monocyte"),
                      ("erythroblast", "erythroblast"), ("platelet", "platelet")]:
        if word in f: return cls
    return None

rows, unmapped = [], {}
for zpath, name in DATASETS:
    if not os.path.exists(zpath):
        print(f"⚠ not found, skipping: {zpath}"); continue
    if os.path.isdir(zpath):
        dest = zpath                      # Kaggle input folder: read in place
    else:
      dest = f"/content/data/{name}"
      if not os.path.isdir(dest):
        print("Unzipping", zpath, "…")
        with zipfile.ZipFile(zpath) as z: z.extractall(dest)
    for p in glob.glob(f"{dest}/**/*", recursive=True):
        if not p.lower().endswith(IMG_EXT) or "__MACOSX" in p: continue
        cls = code_to_class(os.path.dirname(p), p)
        if cls is None:
            k = os.path.basename(os.path.dirname(p)); unmapped[k] = unmapped.get(k, 0) + 1; continue
        rows.append(dict(path=p, cls=cls, source=name))
df = pd.DataFrame(rows)
assert len(df), "No images mapped — check DATASETS paths and LABEL_MAP"
# Keep a generous random sample per class *before* hashing, so huge sets (171k marrow images) stay fast.
df = pd.concat([g.sample(min(len(g), int(MAX_PER_CLASS * 1.3)), random_state=SEED) for _, g in df.groupby("cls")]).reset_index(drop=True)
if unmapped: print("Unmapped folders (ignored):", unmapped)

df["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in df.path]
n0 = len(df); df = df.drop_duplicates("md5"); print(f"Removed {n0-len(df)} exact duplicate images")
conflict = df.groupby("md5").cls.nunique(); assert (conflict <= 1).all()

df = df[df.cls.isin(CLASSES)]
df = pd.concat([g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED) for _, g in df.groupby("cls")]).reset_index(drop=True)
present = [c for c in CLASSES if c in set(df.cls)]
missing = [c for c in CLASSES if c not in present]
if missing: print("⚠ No training images for:", missing, "— the model will never predict these.")

# ---- Split: 70% train / 15% validation / 15% test, stratified by class AND source dataset ----
# Exact duplicates were removed above, so no image can appear in two splits (no leakage).
strat = df.cls + "|" + df.source
rare = strat.map(strat.value_counts()) < 7          # groups too small to stratify jointly fall back to class only
strat = strat.where(~rare, df.cls)
tr, te = train_test_split(df, test_size=TEST_FRAC, stratify=strat, random_state=SEED)
tr, va = train_test_split(tr, test_size=VAL_FRAC / (1 - TEST_FRAC), stratify=strat.loc[tr.index], random_state=SEED)
assert not (set(tr.md5) & set(va.md5) or set(tr.md5) & set(te.md5) or set(va.md5) & set(te.md5))
pd.concat([tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")])[["path", "cls", "source", "md5", "split"]] \
  .to_csv(f"{OUT_DIR}/split_manifest.csv", index=False)
print(f"Split: train {len(tr)} ({len(tr)/len(df):.0%}) | val {len(va)} ({len(va)/len(df):.0%}) | test {len(te)} ({len(te)/len(df):.0%}) — saved split_manifest.csv")
c2i = {c: i for i, c in enumerate(CLASSES)}

class PathDS(Dataset):
    def __init__(s, d, tf): s.p, s.y, s.tf = list(d.path), [c2i[c] for c in d.cls], tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return s.tf(Image.open(s.p[i]).convert("RGB")), s.y[i]

train_ds, val_ds, test_ds = PathDS(tr, train_tf), PathDS(va, eval_tf), PathDS(te, eval_tf)
y_train = np.array(train_ds.y)
summary = pd.DataFrame({"train": tr.cls.value_counts(), "val": va.cls.value_counts(), "test": te.cls.value_counts()}).reindex(CLASSES).fillna(0).astype(int)
print(summary); print(pd.crosstab(df.cls, df.source))
kw = dict(batch_size=BATCH_SIZE, num_workers=min(4, os.cpu_count() or 2), pin_memory=True, persistent_workers=True)
train_dl = DataLoader(train_ds, shuffle=True, drop_last=True, **kw)
val_dl, test_dl = DataLoader(val_ds, **kw), DataLoader(test_ds, **kw)

In [ ]:
# 6. Check the split matches the training run (deployed model metadata: n_train = 30,649; test = 6,568)
print("train", len(tr), "| val", len(va), "| test", len(te))
SPLIT_MATCH = (len(tr) == 30649 and len(te) == 6568)
print("✓ Split sizes match the deployed model's training run" if SPLIT_MATCH else "⚠ Split sizes differ from the original run — results will be reported as an approximate re-split")

In [ ]:
# 7. Download the deployed model (cell_model.onnx in two parts + labels) from the project GitHub repository
import urllib.request, onnxruntime as ort
BASE = "https://raw.githubusercontent.com/drsayantanighoshhazra-jpg/haem5-reporter/main/frontend/model/"
os.makedirs("/content/deployed", exist_ok=True)
for f in ["cell_model.part1.bin", "cell_model.part2.bin", "cell_labels.json"]:
    urllib.request.urlretrieve(BASE + f, "/content/deployed/" + f)
with open("/content/deployed/cell_model.onnx", "wb") as out:
    for f in ["cell_model.part1.bin", "cell_model.part2.bin"]: out.write(open("/content/deployed/" + f, "rb").read())
META = json.load(open("/content/deployed/cell_labels.json"))
assert META["classes"] == CLASSES, "class order differs"
prov = [p for p in ("CUDAExecutionProvider", "CPUExecutionProvider") if p in ort.get_available_providers()]
sess = ort.InferenceSession("/content/deployed/cell_model.onnx", providers=prov)
print("✓ Deployed model loaded:", prov[0], "| blast threshold", META.get("thresholds"))

In [ ]:
# 8. Predict (single view = training-notebook evaluation; 4 rotations averaged = what the app does)
from torch.utils.data import DataLoader
def run(d):
    dl = DataLoader(PathDS(d, eval_tf), batch_size=64, num_workers=2)
    P1, P4, Y = [], [], []
    for x, y in dl:
        xs = x.numpy().astype(np.float32)
        lg = [sess.run(None, {"input": np.ascontiguousarray(np.rot90(xs, k, axes=(2, 3)))})[0] for k in range(4)]
        sm = [np.exp(l - l.max(1, keepdims=True)) / np.exp(l - l.max(1, keepdims=True)).sum(1, keepdims=True) for l in lg]
        P1.append(sm[0]); P4.append(np.mean(sm, 0)); Y.append(np.asarray(y))
    return np.concatenate(P1), np.concatenate(P4), np.concatenate(Y)
t0 = time.time()
vP1, vP4, vY = run(va); tP1, tP4, tY = run(te)
print(f"✓ predicted {len(vY)} validation + {len(tY)} test images in {time.time()-t0:.0f}s")
np.savez_compressed(f"{OUT_DIR}/test_predictions.npz", test_p1=tP1, test_p4=tP4, test_y=tY, val_p1=vP1, val_p4=vP4, val_y=vY,
                    classes=np.array(CLASSES), test_paths=np.array([os.path.relpath(p, "/") for p in te.path]), test_md5=np.array(te.md5))

In [ ]:
# 9. Statistics with bootstrap 95% CIs (2,000 resamples of test images)
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(42); B = 2000; b = CLASSES.index("blast"); THR = (META.get("thresholds") or {}).get("blast", 0.15)
labs = [CLASSES.index(c) for c in present]
def stats(P, Y, thr=None):
    pred = P.argmax(1).copy()
    if thr is not None: pred[P[:, b] >= thr] = b
    tb, pb = Y == b, pred == b
    return dict(accuracy=accuracy_score(Y, pred), balanced_accuracy=balanced_accuracy_score(Y, pred),
                macro_f1=f1_score(Y, pred, average="macro", labels=labs, zero_division=0), kappa=cohen_kappa_score(Y, pred),
                blast_sens=(tb & pb).sum() / max(tb.sum(), 1), blast_spec=(~tb & ~pb).sum() / max((~tb).sum(), 1),
                blast_ppv=(tb & pb).sum() / max(pb.sum(), 1), blast_auc=roc_auc_score(tb, P[:, b]),
                n=len(Y), n_blast=int(tb.sum()), blast_tp=int((tb & pb).sum()), blast_fp=int((~tb & pb).sum()))
def boot(P, Y, thr=None):
    pt = stats(P, Y, thr); keys = [k for k in pt if not k.startswith("n") and k not in ("blast_tp", "blast_fp")]
    sm = {k: [] for k in keys}
    for _ in range(B):
        i = rng.integers(0, len(Y), len(Y)); s = stats(P[i], Y[i], thr)
        for k in keys: sm[k].append(s[k])
    return {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else int(v)) for k, v in pt.items()} | \
           {k + "_ci95": [round(float(np.percentile(sm[k], 2.5)), 4), round(float(np.percentile(sm[k], 97.5)), 4)] for k in keys}
RES = {"split_match": bool(SPLIT_MATCH), "n_train": len(tr), "n_val": len(va), "n_test": len(te), "blast_threshold": THR,
       "single_view_argmax": boot(tP1, tY), "single_view_thr": boot(tP1, tY, THR),
       "rot4_argmax": boot(tP4, tY), "rot4_thr": boot(tP4, tY, THR),
       "per_class_f1_single_view": dict(zip(present, [round(float(x), 4) for x in f1_score(tY, tP1.argmax(1), labels=labs, average=None, zero_division=0)])),
       "false_blast_sources_thr_single_view": pd.Series([CLASSES[t] for t in tY[(tP1[:, b] >= THR) | (tP1.argmax(1) == b)] if t != b]).value_counts().to_dict()}
json.dump(RES, open(f"{OUT_DIR}/test_prediction_stats.json", "w"), indent=2)
print(json.dumps({k: RES[k] for k in ("split_match", "n_test")}, indent=1))
for k in ("single_view_argmax", "single_view_thr", "rot4_argmax", "rot4_thr"):
    r = RES[k]; print(f"{k:20s} acc {r['accuracy']:.3f} {r['accuracy_ci95']}  κ {r['kappa']:.3f} {r['kappa_ci95']}  macroF1 {r['macro_f1']:.3f}  "
                      f"blast sens {r['blast_sens']:.3f} spec {r['blast_spec']:.3f} {r['blast_spec_ci95']} AUC {r['blast_auc']:.3f}")

In [ ]:
# 10. Download the results (send this zip back to Claude)
import shutil
pack = "/content/haem5_test_predictions_results"
os.makedirs(pack, exist_ok=True)
for f in ["test_prediction_stats.json", "test_predictions.npz", "split_manifest.csv"]:
    shutil.copy(f"{OUT_DIR}/{f}", pack)
shutil.make_archive(pack, "zip", pack)
from google.colab import files; files.download(pack + ".zip")
print("✓ Done — send haem5_test_predictions_results.zip to Claude")